In [1]:
# CSV 챗봇 예제에 필요한 라이브러리를 불러옵니다.
from pathlib import Path
import os
from dotenv import load_dotenv, find_dotenv

from llama_index.readers.file import PDFReader
from llama_index.llms.cohere import Cohere
from llama_index.embeddings.cohere import CohereEmbedding
from llama_index.core import Settings, VectorStoreIndex, Document
import pandas as pd

In [2]:
# CSV 경로와 Cohere API 설정을 준비합니다.
CSV_PATH = Path('../Data/ChatbotData.csv')

# Cohere API Key는 Git에 올리지 않기 위해 .env 파일에서 읽습니다.
# 프로젝트 루트에 `.env` 파일을 만들고 아래처럼 저장하세요.
# COHERE_API_KEY=your_cohere_api_key
env_path = find_dotenv(filename='.env', usecwd=True)
load_dotenv(env_path)

cohere_api_key = os.getenv('COHERE_API_KEY')
if not cohere_api_key:
    raise ValueError('COHERE_API_KEY가 없습니다. 프로젝트 루트의 .env 파일을 확인하세요.')

Settings.llm = Cohere(
    # model='command-r-08-2024',
    model='command-r7b-12-2024',
    api_key=cohere_api_key,
    temperature=0,  # 낮을수록 일관된 답변을 생성합니다.
)
Settings.embed_model = CohereEmbedding(
    api_key=cohere_api_key,
    model_name='embed-multilingual-v3.0',
    input_type='search_document',
    embed_batch_size=96,
)

print(f'CSV경로: {CSV_PATH.resolve()}')
print('Cohere / LlamaIndex 설정 완료')

CSV경로: /Users/cheng80/Documents/WorkSpace/RAG/Data/ChatbotData.csv
Cohere / LlamaIndex 설정 완료


#### CSV를 문서 형태로 변환

In [3]:
# 챗봇 학습에 사용할 CSV 파일을 DataFrame으로 읽습니다.
df = pd.read_csv(CSV_PATH)
df.head()

,Q,A,label
0,12시 땡!,하루가 또 가네요.,0
1,1지망 학교 떨어졌어,위로해 드립니다.,0
2,3박4일 놀러가고 싶다,여행은 언제나 좋죠.,0
3,3박4일 정도 놀러가고 싶다,여행은 언제나 좋죠.,0
4,PPL 심하네,눈살이 찌푸려지죠.,0


In [4]:
# 벡터화할 텍스트 컬럼과 metadata 컬럼을 지정합니다.
TEXT_COLUMNS = ['Q','A']
METADATA_COLUMNS = ['label']

MAX_ROWS = 1000
df = df.head(MAX_ROWS).copy()

display(df.head())
print('문서와 대상 컬럼 : ',TEXT_COLUMNS)
print('메타데이터 컬럼 : ',METADATA_COLUMNS)
print('사용할 행수 : ',MAX_ROWS)


,Q,A,label
0,12시 땡!,하루가 또 가네요.,0
1,1지망 학교 떨어졌어,위로해 드립니다.,0
2,3박4일 놀러가고 싶다,여행은 언제나 좋죠.,0
3,3박4일 정도 놀러가고 싶다,여행은 언제나 좋죠.,0
4,PPL 심하네,눈살이 찌푸려지죠.,0


문서와 대상 컬럼 :  ['Q', 'A']
메타데이터 컬럼 :  ['label']
사용할 행수 :  1000


In [5]:
# CSV의 각 행을 LlamaIndex Document 객체로 변환합니다.
# 각 Row를 질문-답변 형태의 문서로 변환

def row_to_document(row:pd.Series, row_number:int) -> Document:
    text_parts = []

    for column in TEXT_COLUMNS:
        value = row[column]

        if pd.isna(value):
            continue
        text_parts.append(f'{column}:{value}')

    metadata = {
        'row_number' : row_number,
        'label' : row['label']
    }
    return Document(
        text = ' | '.join(text_parts),
        metadata = metadata
    )    
# DataFrame의 각 row를 Document로 변환
documents = [row_to_document(row, idx) for idx, row in df.iterrows()]

print('생성된 Document수 : ',len(documents))
print('첫번째 Document 예제')
print(documents[0].text)

생성된 Document수 :  1000
첫번째 Document 예제
Q:12시 땡! | A:하루가 또 가네요.


In [6]:
# 변환한 문서들로 벡터 인덱스와 chat engine을 생성합니다.
# 문서목록으로 벡터 인덱스 생성
index = VectorStoreIndex.from_documents(documents)

# 검색된 문서를 바탕으로 답변하는 chat engine을 제작
# as_query_engine는 검색하여 답변
# as_chat_engine은 추론
chat_engine = index.as_chat_engine(
    chat_mode='context',
    similartity_top_k = 5,
    verbose = True
)

2026-06-02 11:39:19,169 - INFO - HTTP Request: POST https://api.cohere.com/v2/embed "HTTP/1.1 200 OK"
2026-06-02 11:39:20,325 - INFO - HTTP Request: POST https://api.cohere.com/v2/embed "HTTP/1.1 200 OK"
2026-06-02 11:39:21,460 - INFO - HTTP Request: POST https://api.cohere.com/v2/embed "HTTP/1.1 200 OK"
2026-06-02 11:39:22,480 - INFO - HTTP Request: POST https://api.cohere.com/v2/embed "HTTP/1.1 200 OK"
2026-06-02 11:39:23,167 - INFO - HTTP Request: POST https://api.cohere.com/v2/embed "HTTP/1.1 200 OK"
2026-06-02 11:39:24,318 - INFO - HTTP Request: POST https://api.cohere.com/v2/embed "HTTP/1.1 200 OK"
2026-06-02 11:39:25,269 - INFO - HTTP Request: POST https://api.cohere.com/v2/embed "HTTP/1.1 200 OK"
2026-06-02 11:39:25,929 - INFO - HTTP Request: POST https://api.cohere.com/v2/embed "HTTP/1.1 200 OK"
2026-06-02 11:39:26,419 - INFO - HTTP Request: POST https://api.cohere.com/v2/embed "HTTP/1.1 200 OK"
2026-06-02 11:39:27,456 - INFO - HTTP Request: POST https://api.cohere.com/v2/embe

In [7]:
# 고정 질문으로 chat engine 동작을 테스트합니다.
# Test
question = '12시 땡! 이라는 질문에는 어떤 답변이 연결되어 있어?'
response = chat_engine.chat(question)
print('질문:',question)
print('응답:',response)

2026-06-02 11:39:28,656 - INFO - HTTP Request: POST https://api.cohere.com/v2/embed "HTTP/1.1 200 OK"
2026-06-02 11:39:29,311 - INFO - HTTP Request: POST https://api.cohere.com/v1/chat "HTTP/1.1 200 OK"


질문: 12시 땡! 이라는 질문에는 어떤 답변이 연결되어 있어?
응답: "하루가 또 가네요."라는 답변이 "12시 땡!"이라는 질문에 연결되어 있습니다.


In [8]:
# 사용자가 exit 또는 quit를 입력할 때까지 질문을 반복해서 받습니다.
# 여러번 질문하고
# exit, quit 종료
while True:
    user_question = input('질문을 입력하세요:').strip()
    if user_question.lower() in {'exit',quit}:
        print('쳇봇을 종료합니다.')
        break
    if not user_question:
        print('빈 질문은 처리 할 수 없다. 다시 입력하여라')
        continue

    answer = chat_engine.chat(user_question)
    print('\n[응답]')
    print(answer)
    print('-'*50)    


2026-06-02 11:39:46,233 - INFO - HTTP Request: POST https://api.cohere.com/v2/embed "HTTP/1.1 200 OK"
2026-06-02 11:39:46,784 - INFO - HTTP Request: POST https://api.cohere.com/v1/chat "HTTP/1.1 200 OK"



[응답]
12시 땡! 하루가 또 가네요. 내일도 힘내세요!
--------------------------------------------------
빈 질문은 처리 할 수 없다. 다시 입력하여라
쳇봇을 종료합니다.
